In [ ]:
from decimal import Decimal, ROUND_HALF_UP


def validate_heatmap_inputs(results, metrics, model_order):
    required_columns = {"threshold", "model", *metrics}
    missing_columns = sorted(required_columns - set(results.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    if results["model"].nunique() != 2:
        raise ValueError("This heatmap requires exactly two models.")

    if model_order is not None and set(model_order) != set(results["model"].unique()):
        raise ValueError("model_order must match the two model names in results.")

    duplicate_rows = results.duplicated(subset=["threshold", "model"]).any()
    if duplicate_rows:
        raise ValueError("Each threshold-model combination must appear only once.")


def resolve_model_order(results, model_order=None):
    if model_order is not None:
        return model_order
    return sorted(results["model"].unique().tolist())


def build_metric_table(results, metric, model_order):
    metric_table = results.pivot(
        index="threshold",
        columns="model",
        values=metric,
    )
    return metric_table[model_order].sort_index()


def quantize_metric_value(value, decimals=3):
    if pd.isna(value):
        return None

    quantizer = Decimal("1").scaleb(-decimals)
    return Decimal(str(float(value))).quantize(quantizer, rounding=ROUND_HALF_UP)


def format_metric_value(value, decimals=3):
    quantized_value = quantize_metric_value(value, decimals=decimals)
    if quantized_value is None:
        return "NA"
    return format(quantized_value, f".{decimals}f")


def choose_winner_from_values(
    value_a,
    value_b,
    model_a,
    model_b,
    higher_is_better,
    tie_decimals=3,
):
    if pd.isna(value_a) and pd.isna(value_b):
        return "Tie"
    if pd.isna(value_a):
        return model_b
    if pd.isna(value_b):
        return model_a
    rounded_a = quantize_metric_value(value_a, decimals=tie_decimals)
    rounded_b = quantize_metric_value(value_b, decimals=tie_decimals)
    if rounded_a == rounded_b:
        return "Tie"

    if higher_is_better:
        return model_a if rounded_a > rounded_b else model_b
    return model_a if rounded_a < rounded_b else model_b


def build_winner_table(results, metrics, metric_directions, model_order, tie_decimals=3):
    winner_columns = {}

    for metric in metrics:
        metric_table = build_metric_table(results, metric, model_order)
        model_a, model_b = model_order

        winners = metric_table.apply(
            lambda row: choose_winner_from_values(
                value_a=row[model_a],
                value_b=row[model_b],
                model_a=model_a,
                model_b=model_b,
                higher_is_better=metric_directions[metric],
                tie_decimals=tie_decimals,
            ),
            axis=1,
        )
        winner_columns[metric] = winners

    winner_table = pd.DataFrame(winner_columns)
    winner_table.index.name = "threshold"
    return winner_table


def build_annotation_table(results, metrics, model_order, model_labels=None, annotation_decimals=3):
    model_labels = model_labels or {}
    annotations = {}

    for metric in metrics:
        metric_table = build_metric_table(results, metric, model_order)
        model_a, model_b = model_order
        label_a = model_labels.get(model_a, model_a)
        label_b = model_labels.get(model_b, model_b)

        annotations[metric] = metric_table.apply(
            lambda row: (
                f"{label_a}: {format_metric_value(row[model_a], decimals=annotation_decimals)}\n"
                f"{label_b}: {format_metric_value(row[model_b], decimals=annotation_decimals)}"
            ),
            axis=1,
        )

    annotation_table = pd.DataFrame(annotations)
    annotation_table.index.name = "threshold"
    return annotation_table


def prettify_metric_labels(metrics):
    return [metric.replace("_", " ").title() for metric in metrics]


def plot_threshold_winner_heatmap(
    results,
    metrics,
    metric_directions,
    title="Which Model Performs Better Across Decision Thresholds?",
    model_order=None,
    model_labels=None,
    metric_labels=None,
    output_path=None,
    figsize=(12, 7),
    tie_decimals=3,
):
    validate_heatmap_inputs(results, metrics, model_order)
    model_order = resolve_model_order(results, model_order)
    model_a, model_b = model_order

    winner_table = build_winner_table(
        results=results,
        metrics=metrics,
        metric_directions=metric_directions,
        model_order=model_order,
        tie_decimals=tie_decimals,
    )

    annotation_table = build_annotation_table(
        results=results,
        metrics=metrics,
        model_order=model_order,
        model_labels=model_labels,
        annotation_decimals=tie_decimals,
    )

    code_map = {model_a: 0, model_b: 1, "Tie": 2}
    encoded = winner_table.apply(lambda col: col.map(code_map)).astype(int)

    cmap = ListedColormap(["#4C78A8", "#F58518", "#BDBDBD"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

    sns.set_theme(style="white", font_scale=0.95)
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        encoded,
        annot=annotation_table,
        fmt="",
        cmap=cmap,
        norm=norm,
        cbar=False,
        linewidths=1,
        linecolor="white",
        annot_kws={"fontsize": 9, "fontweight": "semibold"},
        ax=ax,
    )

    display_labels = model_labels or {}
    x_labels = metric_labels if metric_labels is not None else prettify_metric_labels(metrics)
    y_labels = [f"{float(threshold):.2%}" for threshold in encoded.index]

    ax.set_title(title, fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Metric", fontsize=11)
    ax.set_ylabel("Decision Threshold", fontsize=11)
    ax.set_xticklabels(x_labels, rotation=0, ha="center")
    ax.set_yticklabels(y_labels, rotation=0)

    ax.legend(
        handles=[
            mpatches.Patch(color="#4C78A8", label=f"{display_labels.get(model_a, model_a)} wins"),
            mpatches.Patch(color="#F58518", label=f"{display_labels.get(model_b, model_b)} wins"),
            mpatches.Patch(color="#BDBDBD", label="Tie"),
        ],
        title="Better Model",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=3,
        frameon=False,
    )

    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return winner_table, annotation_table, fig


def plot_flip_analysis_by_error_type(
    model_outputs,
    title="Decision Flip Analysis by Error Type",
    figsize=(14, 6),
    output_path=None,
):
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt

    required_keys = {
        "test_target",
        "linear_predictions",
        "random_forest_predictions",
        "detailed_results",
    }
    missing_keys = sorted(required_keys - set(model_outputs))
    if missing_keys:
        raise ValueError(f"model_outputs is missing required keys: {missing_keys}")

    thresholds = sorted(model_outputs["detailed_results"]["threshold"].unique())
    plot_base = pd.DataFrame({
        "actual_return": pd.Series(model_outputs["test_target"]).reset_index(drop=True),
        "linear_prediction": pd.Series(model_outputs["linear_predictions"]).reset_index(drop=True),
        "random_forest_prediction": pd.Series(model_outputs["random_forest_predictions"]).reset_index(drop=True),
    })

    direction_order = [
        "LR approve / RF reject",
        "RF approve / LR reject",
    ]
    error_type_order = ["Type I Error", "Type II Error"]
    error_model_lookup = {
        ("LR approve / RF reject", "Type I Error"): "LinearRegression",
        ("LR approve / RF reject", "Type II Error"): "RandomForest",
        ("RF approve / LR reject", "Type I Error"): "RandomForest",
        ("RF approve / LR reject", "Type II Error"): "LinearRegression",
    }

    flip_records = []
    for threshold in thresholds:
        linear_approve = plot_base["linear_prediction"] >= threshold
        random_forest_approve = plot_base["random_forest_prediction"] >= threshold
        disagreement_mask = linear_approve != random_forest_approve

        if not disagreement_mask.any():
            continue

        flipped = plot_base.loc[disagreement_mask, ["actual_return"]].copy()
        flipped["threshold"] = threshold
        flipped["flip_direction"] = np.where(
            linear_approve.loc[disagreement_mask],
            "LR approve / RF reject",
            "RF approve / LR reject",
        )
        flipped["error_type"] = np.where(
            flipped["actual_return"] < 0,
            "Type I Error",
            "Type II Error",
        )
        flipped["error_model"] = flipped.apply(
            lambda row: error_model_lookup[(row["flip_direction"], row["error_type"])],
            axis=1,
        )
        flip_records.append(flipped)

    summary_columns = [
        "threshold",
        "threshold_label",
        "flip_direction",
        "error_type",
        "error_model",
        "flip_count",
        "direction_share",
        "threshold_share",
    ]

    if not flip_records:
        empty_summary = pd.DataFrame(columns=summary_columns)
        fig, ax = plt.subplots(figsize=figsize)
        ax.text(0.5, 0.5, "No decision flips found.", ha="center", va="center", fontsize=12)
        ax.axis("off")
        if output_path is not None:
            fig.savefig(output_path, dpi=300, bbox_inches="tight")
        return empty_summary, fig

    flip_errors = pd.concat(flip_records, ignore_index=True)

    summary = (
        flip_errors.groupby(["threshold", "flip_direction", "error_type"], as_index=False)
        .size()
        .rename(columns={"size": "flip_count"})
    )

    full_index = pd.MultiIndex.from_product(
        [thresholds, direction_order, error_type_order],
        names=["threshold", "flip_direction", "error_type"],
    )
    summary = (
        summary.set_index(["threshold", "flip_direction", "error_type"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    summary["error_model"] = summary.apply(
        lambda row: error_model_lookup[(row["flip_direction"], row["error_type"])],
        axis=1,
    )
    summary["threshold_label"] = summary["threshold"].map(lambda value: f"{float(value):.2%}")

    direction_totals = summary.groupby(["threshold", "flip_direction"])["flip_count"].transform("sum")
    threshold_totals = summary.groupby("threshold")["flip_count"].transform("sum")

    summary["direction_share"] = np.where(
        direction_totals > 0,
        summary["flip_count"] / direction_totals,
        np.nan,
    )
    summary["threshold_share"] = np.where(
        threshold_totals > 0,
        summary["flip_count"] / threshold_totals,
        np.nan,
    )
    summary = summary[summary_columns].copy()

    sns.set_theme(style="whitegrid", font_scale=0.95)
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)
    palette = {
        "Type I Error": "#D95F02",
        "Type II Error": "#4C78A8",
    }

    for ax, direction in zip(axes, direction_order):
        direction_data = summary.loc[summary["flip_direction"] == direction].copy()
        sns.barplot(
            data=direction_data,
            x="threshold_label",
            y="flip_count",
            hue="error_type",
            hue_order=error_type_order,
            palette=palette,
            ax=ax,
        )

        ax.set_title(direction, fontsize=12, fontweight="bold")
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Flipped Observations" if direction == direction_order[0] else "")
        ax.tick_params(axis="x", rotation=0)

        for container in ax.containers:
            labels = [
                f"{int(round(bar.get_height()))}" if bar.get_height() > 0 else ""
                for bar in container
            ]
            ax.bar_label(container, labels=labels, padding=3, fontsize=9)

        legend = ax.get_legend()
        if direction == direction_order[0]:
            ax.legend(title="Error Type")
        elif legend is not None:
            legend.remove()

    fig.suptitle(
        title + "\nType I: approve negative return | Type II: reject non-negative return",
        fontsize=14,
        fontweight="bold",
        y=1.03,
    )
    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return summary, fig
